In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# ============================================================
# 1. Load Subscription CDC Work Table
# ============================================================

subscription_cdc_df = spark.table(
    "customer360_dev.control.subscription_cdc_work"
)

target_table = "customer360_dev.silver.subscriptions"

silver_delta = DeltaTable.forName(
    spark,
    target_table
)


# ============================================================
# 2. Count CDC Actions
# ============================================================

insert_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "INSERT")
    .count()
)

update_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "UPDATE")
    .count()
)

unchanged_count = (
    subscription_cdc_df
    .filter(F.col("cdc_action") == "UNCHANGED")
    .count()
)

print(f"INSERT    : {insert_count:,}")
print(f"UPDATE    : {update_count:,}")
print(f"UNCHANGED : {unchanged_count:,}")

INSERT    : 0
UPDATE    : 0
UNCHANGED : 0


In [0]:
# ============================================================
# 3. Capture Silver Count Before Merge
# ============================================================

silver_before_count = (
    spark.table(target_table)
    .count()
)

print(
    f"Silver Subscriptions before merge: "
    f"{silver_before_count:,}"
)

Silver Subscriptions before merge: 978,054


In [0]:
# ============================================================
# 4. Keep Only INSERT / UPDATE Rows
# ============================================================

merge_source_df = (
    subscription_cdc_df
    .filter(
        F.col("cdc_action").isin(
            "INSERT",
            "UPDATE"
        )
    )
)

In [0]:
(
    silver_delta.alias("tgt")

    .merge(
        merge_source_df.alias("src"),
        "tgt.subscription_id = src.subscription_id"
    )

    .whenMatchedUpdate(
        condition="""
            src.cdc_action = 'UPDATE'
            AND (
                tgt.updated_at IS NULL
                OR src.updated_at >= tgt.updated_at
            )
        """,

        set={
            "customer_id":
                "src.customer_id",

            "plan_code":
                "src.plan_code",

            "subscription_status":
                "src.subscription_status",

            "start_date":
                "src.start_date",

            "end_date":
                "src.end_date",

            "billing_cycle":
                "src.billing_cycle",

            "monthly_fee":
                "src.monthly_fee",

            "auto_renew":
                "src.auto_renew",

            "payment_method_type":
                "src.payment_method_type",

            "trial_flag":
                "src.trial_flag",

            "cancel_reason":
                "src.cancel_reason",

            "created_at":
                "src.created_at",

            "updated_at":
                "src.updated_at",

            "_source_file":
                "src._source_file",

            "_source_system":
                "src._source_system",

            "_load_type":
                "src._load_type",

            "_ingestion_timestamp":
                "src._ingestion_timestamp",

            "_silver_processed_at":
                "current_timestamp()"
        }
    )

    .whenNotMatchedInsert(
        condition="""
            src.cdc_action = 'INSERT'
        """,

        values={
            "subscription_id":
                "src.subscription_id",

            "customer_id":
                "src.customer_id",

            "plan_code":
                "src.plan_code",

            "subscription_status":
                "src.subscription_status",

            "start_date":
                "src.start_date",

            "end_date":
                "src.end_date",

            "billing_cycle":
                "src.billing_cycle",

            "monthly_fee":
                "src.monthly_fee",

            "auto_renew":
                "src.auto_renew",

            "payment_method_type":
                "src.payment_method_type",

            "trial_flag":
                "src.trial_flag",

            "cancel_reason":
                "src.cancel_reason",

            "created_at":
                "src.created_at",

            "updated_at":
                "src.updated_at",

            "_source_file":
                "src._source_file",

            "_source_system":
                "src._source_system",

            "_load_type":
                "src._load_type",

            "_ingestion_timestamp":
                "src._ingestion_timestamp",

            "_silver_processed_at":
                "current_timestamp()"
        }
    )

    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
silver_after_count = (
    spark.table(target_table)
    .count()
)

actual_growth = (
    silver_after_count
    - silver_before_count
)

print(f"Silver before   : {silver_before_count:,}")
print(f"Silver after    : {silver_after_count:,}")
print(f"Actual growth   : {actual_growth:,}")
print(f"Expected inserts: {insert_count:,}")

Silver before   : 978,054
Silver after    : 978,054
Actual growth   : 0
Expected inserts: 0


In [0]:
duplicate_subscription_ids = (
    spark.table(target_table)
    .groupBy("subscription_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate subscription IDs in Silver:",
    duplicate_subscription_ids
)

Duplicate subscription IDs in Silver: 0


In [0]:
orphan_subscriptions = (
    spark.table(target_table)
    .join(
        spark.table(
            "customer360_dev.silver.customers"
        ).select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

print(
    "Subscriptions with missing Silver Customer:",
    orphan_subscriptions
)

Subscriptions with missing Silver Customer: 0
